## Python

### Проверка подключения

In [ ]:
# Подключение к Redis
import redis

In [ ]:
# Создаем подключение
r = redis.Redis(
    host='redis',
    port=6379,
    password='redispass',
    decode_responses=True  # автоматически декодировать ответы в строки
)

In [ ]:
# Проверка подключения
try:
    response = r.ping()
    print(f"Redis connection successful: {response}")
except redis.ConnectionError as e:
    print(f"Connection failed: {e}")

### Базовые операции

#### Строки

In [ ]:
# 1. Работа со строками (Strings)
print("=== STRING OPERATIONS ===")
r.set('user:1000:name', 'John Doe')
r.set('user:1000:email', 'john@example.com')
r.set('user:1000:age', 30)

In [ ]:
# Получаем значения
name = r.get('user:1000:name')
email = r.get('user:1000:email')
age = r.get('user:1000:age')

print(f"User: {name}, Email: {email}, Age: {age}")

#### Инкремент

In [ ]:
# Инкремент
r.incr('user:1000:visits')
r.incrby('user:1000:visits', 5)
visits = r.get('user:1000:visits')
print(f"Visits: {visits}")

#### Списки

In [ ]:
# 2. Работа со списками (Lists)
print("\n=== LIST OPERATIONS ===")
# Добавляем задачи в очередь
r.lpush('tasks:queue', 'task1', 'task2', 'task3')
r.rpush('tasks:queue', 'task4')

In [ ]:
# Получаем все задачи
tasks = r.lrange('tasks:queue', 0, -1)
print(f"All tasks: {tasks}")

In [ ]:
# Извлекаем задачу (как очередь)
task = r.lpop('tasks:queue')
print(f"Processed task: {task}")

#### Множества

In [ ]:
# 3. Работа с множествами (Sets)
print("\n=== SET OPERATIONS ===")
r.sadd('users:active', 'user1', 'user2', 'user3', 'user1')  # дубликаты игнорируются
active_users = r.smembers('users:active')
print(f"Active users: {active_users}")

r.sadd('users:premium', 'user2', 'user3', 'user4')
premium_users = r.smembers('users:premium')
print(f"Premium users: {premium_users}")

# Пересечение множеств
common_users = r.sinter('users:active', 'users:premium')
print(f"Users in both sets: {common_users}")

#### Хэш-таблицы

In [ ]:
# 4. Работа с хеш-таблицами (Hashes)
print("\n=== HASH OPERATIONS ===")
user_data = {
    'name': 'Alice Smith',
    'email': 'alice@example.com',
    'department': 'Engineering',
    'salary': '75000'
}
r.hset('user:2000', mapping=user_data)

In [ ]:
# Получаем все поля
all_fields = r.hgetall('user:2000')
print(f"User 2000 data: {all_fields}")

In [ ]:
# Получаем конкретные поля
name = r.hget('user:2000', 'name')
email = r.hget('user:2000', 'email')
print(f"Name: {name}, Email: {email}")

#### Сортированные множества

In [ ]:
# 5. Работа с отсортированными множествами (Sorted Sets)
print("\n=== SORTED SET OPERATIONS ===")
# Добавляем игроков с их счетами
r.zadd('game:scores', {
    'player1': 100,
    'player2': 250,
    'player3': 150,
    'player4': 300
})

In [ ]:
# Топ-3 игрока
top_players = r.zrevrange('game:scores', 0, 2, withscores=True)
print(f"Top players: {top_players}")

# Количество игроков со счетом > 200
count = r.zcount('game:scores', 200, '+inf')
print(f"Players with score > 200: {count}")

# Разобраться

## Использование Redis как кэша

In [ ]:
import time
import json

# Функция с кэшированием
def get_expensive_data(user_id):
    # Проверяем кэш
    cached = r.get(f"cache:user:{user_id}:data")
    if cached:
        print(f"Cache HIT for user {user_id}")
        return json.loads(cached)
    
    print(f"Cache MISS for user {user_id}")
    # Имитация дорогой операции
    time.sleep(2)
    data = {
        'user_id': user_id,
        'transactions': [100, 200, 150],
        'balance': 1000,
        'timestamp': time.time()
    }
    
    # Сохраняем в кэш на 30 секунд
    r.setex(
        f"cache:user:{user_id}:data",
        30,  # TTL в секундах
        json.dumps(data)
    )
    return data

# Тестируем кэширование
print("\n=== CACHING DEMO ===")
start = time.time()
data1 = get_expensive_data(101)
print(f"First call took: {time.time() - start:.2f}s")

start = time.time()
data2 = get_expensive_data(101)
print(f"Second call took: {time.time() - start:.2f}s")
print(f"Data from cache: {data2}")

## Pub/Sub в Redis

In [ ]:
import threading
import time

# Функция для подписки на каналы
def subscriber_thread():
    pubsub = r.pubsub()
    pubsub.subscribe('notifications', 'alerts')
    
    print("Subscriber started. Waiting for messages...")
    for message in pubsub.listen():
        if message['type'] == 'message':
            print(f"[{message['channel']}] Received: {message['data']}")
        if message['data'] == 'stop':
            break

# Запускаем подписчика в отдельном потоке
sub_thread = threading.Thread(target=subscriber_thread, daemon=True)
sub_thread.start()

# Даем время подписчику запуститься
time.sleep(1)

# Публикуем сообщения
print("\n=== PUB/SUB DEMO ===")
r.publish('notifications', 'User logged in')
r.publish('notifications', 'New message received')
r.publish('alerts', 'High CPU usage detected')
r.publish('notifications', 'stop')

time.sleep(2)

# Spark

In [ ]:
from pyspark.sql import SparkSession

drivers = [
    # ==================== REDIS ====================
    # Spark Redis Connector
        "/opt/spark/external-jars/spark-redis_2.12-3.1.0.jar",
    # Jedis - Redis Java client
        "/opt/spark/external-jars/jedis-3.9.0.jar",
    # Commons Pool2 - connection pooling
        "/opt/spark/external-jars/commons-pool2-2.11.1.jar",
    # SLF4J (Simple Logging Facade for Java)
        "/opt/spark/external-jars/slf4j-api-1.7.30.jar",
]

spark = (SparkSession.builder
         .appName("redis-spark")
         .master("spark://spark-master:7077")
         .config("spark.jars", ",".join(drivers))
         .config("spark.redis.host", "redis")
         .config("spark.redis.port", "6379")
         .config("spark.redis.auth", "redispass")
         .getOrCreate()
        )

In [ ]:
from pyspark.sql import Row

# Создаем очень простой DataFrame
simple_data = [
    Row(key="1", value="Иванов Иван Иваныч"),
    Row(key="2", value="Упячка"),
    Row(key="3", value="Тридесятое царство")
]

simple_df = spark.createDataFrame(simple_data)
print("Создан DataFrame:")
simple_df.show()

# Пробуем записать в Redis
try:
    simple_df.write \
        .format("org.apache.spark.sql.redis") \
        .option("table", "spark_write_test") \
        .option("key.column", "key") \
        .mode("overwrite") \
        .save()
    print("✅ Запись в Redis выполнена")
except Exception as e:
    print(f"❌ Ошибка при записи: {e}")

In [ ]:
# Сначала запишем что-то через Python
r.hset("py:user:1", mapping={"name": "Python User", "age": "25", "city": "Moscow"})
r.hset("py:user:2", mapping={"name": "Spark User", "age": "30", "city": "SPb"})

print("Данные записаны через Python")

In [ ]:
# Теперь попробуем прочитать через Spark
try:
    py_data = spark.read \
        .format("org.apache.spark.sql.redis") \
        .option("keys.pattern", "py:user:*") \
        .option("infer.schema", "true") \
        .load()
    
    print("Прочитано в Spark:")
    py_data.show()
except Exception as e:
    print(f"Ошибка чтения: {e}")

## Тысты

In [1]:
import redis

In [2]:
import json

In [3]:
# Параметры подключения (как в docker-compose)

# Создаем подключение
r = redis.Redis(host='redis', port=6379, password='redispass')

# ,
#                 decode_responses=True  # автоматически декодировать ответы в строки
# )

In [4]:
# Проверка подключения
try:
    response = r.ping()
    print(f"Redis connection successful: {response}")
except redis.ConnectionError as e:
    print(f"Connection failed: {e}")

Redis connection successful: True


In [5]:
# Сложные данные нужно сериализовать
user_preferences = {
    "theme": "dark",
    "notifications": True,
    "favorite_colors": ["blue", "green"],
    "layout": {
        "sidebar": True,
        "density": "comfortable"
    }
}

In [6]:
print(user_preferences)

{'theme': 'dark', 'notifications': True, 'favorite_colors': ['blue', 'green'], 'layout': {'sidebar': True, 'density': 'comfortable'}}


In [9]:
for key, value in user_preferences.items():
    print(f"{key}: {value}")

theme: dark
notifications: True
favorite_colors: ['blue', 'green']
layout: {'sidebar': True, 'density': 'comfortable'}


In [11]:
# ⭐⭐⭐ СЕРИАЛИЗАЦИЯ ⭐⭐⭐
json_string = json.dumps(user_preferences)
# dumps = dump string (выгрузить в строку)

print("До сериализации (Python объект):")
print(user_preferences)
print(f"Тип: {type(user_preferences)}")  # <class 'dict'>

print("\nПосле сериализации (JSON строка):")
print(json_string)
print(f"Тип: {type(json_string)}")  # <class 'str'>

До сериализации (Python объект):
{'theme': 'dark', 'notifications': True, 'favorite_colors': ['blue', 'green'], 'layout': {'sidebar': True, 'density': 'comfortable'}}
Тип: <class 'dict'>

После сериализации (JSON строка):
{"theme": "dark", "notifications": true, "favorite_colors": ["blue", "green"], "layout": {"sidebar": true, "density": "comfortable"}}
Тип: <class 'str'>


In [10]:
# Сохраняем в Redis
r.set("sample_user:1001:prefs", json.dumps(user_preferences))

True

In [15]:
# Читаем из Redis
prefs_json = r.get("sample_user:1001:prefs")

In [16]:
print(prefs_json)

b'{"theme": "dark", "notifications": true, "favorite_colors": ["blue", "green"], "layout": {"sidebar": true, "density": "comfortable"}}'


In [17]:
prefs = json.loads(prefs_json)
print(prefs["theme"])  # dark
print(prefs)

dark
{'theme': 'dark', 'notifications': True, 'favorite_colors': ['blue', 'green'], 'layout': {'sidebar': True, 'density': 'comfortable'}}


In [18]:
r.hset("trifid:day", mapping={
    "name": "John",
    "age": "30",
    "email": "john@example.com",
    "scores": json.dumps([95, 87, 92])  # список всё равно сериализуем
})

4